# 01 — Bronze: Raw ingestion

Reads the Kaggle `flights_sample_3m.csv` from the raw volume, writes a Delta
table with an ingestion timestamp, and reports row counts. Idempotent: uses a
deterministic full overwrite so a rerun leaves the same row count.

**Run order:** `01_bronze` → `02_eda` → `03_silver` → `04_gold` → `05_train` → `06_api_ingest` → `07_score`.

## Environment
Requires serverless **environment version 4** (needed later for `pyspark.ml` in `05_train`).

In [ ]:
import sys
sys.path.append("..")  # so `import src` works from /Workspace Git folder

from pyspark.sql.functions import current_timestamp
from src import config

## Read the source CSV

In [ ]:
bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(config.SOURCE_CSV)
    .withColumn("bronze_ingested_at", current_timestamp())
)

bronze_count = bronze_df.count()
print(f"Source rows: {bronze_count:,}")
print(f"Columns:     {len(bronze_df.columns)}")

## Write Bronze Delta table (overwrite = idempotent)

In [ ]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.BRONZE)
)

print(f"Wrote {config.BRONZE}")

## Verify

In [ ]:
verified = spark.table(config.BRONZE)
print(f"Bronze rows: {verified.count():,}")
verified.printSchema()

In [ ]:
%sql
DESCRIBE HISTORY workspace.flights.bronze_flights;